In [ ]:
import pandas as pd
import numpy as np
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)
log = logging.getLogger("pipeline_fraude")

CARPETA = '/content/drive/MyDrive/Colab Notebooks/Gestion_Datos_IA_EV2'
df_clean = pd.read_csv(f'{CARPETA}/datos_limpios.csv')
log.info(f"Dataset cargado: {df_clean.shape}")

errores = []

# --- Validación estructural ---
if df_clean.isnull().sum().sum() > 0:
    errores.append(f"Hay {df_clean.isnull().sum().sum()} valores nulos")

if df_clean['amt'].min() <= 0:
    errores.append(f"Hay montos <= 0: mínimo = {df_clean['amt'].min()}")

if not df_clean['hour'].between(0, 23).all():
    errores.append("Hora fuera de rango 0-23")

if not df_clean['is_night'].isin([0, 1]).all():
    errores.append("is_night tiene valores distintos a 0 y 1")

if not df_clean['is_fraud'].isin([0, 1]).all():
    errores.append("is_fraud tiene valores distintos a 0 y 1")

if df_clean['geo_distance'].min() < 0:
    errores.append("geo_distance tiene valores negativos")

log.info("Validación estructural completada")

# --- Validación semántica ---
amt_fraud = df_clean[df_clean['is_fraud']==1]['amt'].mean()
amt_legit = df_clean[df_clean['is_fraud']==0]['amt'].mean()
if amt_fraud <= amt_legit:
    errores.append("Fraudes no tienen mayor monto promedio")
log.info(f"Monto promedio — fraude: ${amt_fraud:.2f} | legítimo: ${amt_legit:.2f}")

dist_fraud = df_clean[df_clean['is_fraud']==1]['geo_distance'].mean()
dist_legit = df_clean[df_clean['is_fraud']==0]['geo_distance'].mean()
if dist_fraud <= dist_legit:
    errores.append("Fraudes no tienen mayor distancia promedio")
log.info(f"Distancia promedio — fraude: {dist_fraud:.2f} km | legítimo: {dist_legit:.2f} km")

night_fraud = df_clean[df_clean['is_fraud']==1]['is_night'].mean()
night_legit = df_clean[df_clean['is_fraud']==0]['is_night'].mean()
if night_fraud <= night_legit:
    errores.append("Fraudes no ocurren más de noche que legítimas")
log.info(f"Proporción nocturna — fraude: {night_fraud:.2%} | legítimo: {night_legit:.2%}")

proporcion_fraude = df_clean['is_fraud'].mean()
log.info(f"Proporción de fraudes: {proporcion_fraude:.2%}")
if proporcion_fraude < 0.02 or proporcion_fraude > 0.40:
    errores.append(f"Proporción de fraudes inusual: {proporcion_fraude:.2%}")

# --- Resultado ---
if errores:
    for e in errores:
        log.error(f"FALLO: {e}")
    raise ValueError(f"Validación fallida con {len(errores)} error(es): {errores}")
else:
    log.info("✓ Todas las validaciones pasaron correctamente")
    print("\n✓ Validación completada sin errores — datos listos para entrenar")


✓ Validación completada sin errores — datos listos para entrenar
